In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction --extra-index-url https://download.easydiffraction.org/pypi/simple/

# Rietveld fit: Si, SEPD

Fit silicon to time-of-flight neutron powder data from SEPD at Argonne, with the Jorgensen-Von
Dreele profile.

The project is the catalogue case `pd-neut-tof_si-sepd`, the one the command line and the app's
Examples open.

<span style="color:#398648;background:#39864821;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Fitting</span> <span style="color:#96601d;background:#96601d21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Single</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">pd</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">neut</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">tof</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Bragg</span> <span style="color:#00796b;background:#00796b21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Crystal</span>

In [ ]:
import easydiffraction as edi

## Load the project

In [ ]:
project_path = edi.download_data('pd-neut-tof_si-sepd', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['si']
experiment = project.experiments['sepd']

The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [ ]:
for parameter in project.free_parameters:
    parameter.free = False

## Estimate the background

Start the background from an automatic estimate. The default method, arPLS, fits a smooth curve
under the pattern that ignores the peaks, and places 14 points along it.

In [ ]:
experiment.background.auto_estimate(n_points=14)

In [ ]:
project.display.pattern('sepd')

## Perform the analysis

The fit follows diffraction-lib's four cycles for this case, then frees the background once more
so it ends with the case's free set.

### Cell, scale and time-of-flight calibration

In [ ]:
structure.cell.length_a.free = True
experiment.linked_structures['si'].scale.free = True
experiment.instrument.calib_d_to_tof_offset.free = True

In [ ]:
project.display.parameters.free()

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('sepd')

### Background

The estimated points are a starting point; fit them now that the peaks are in place.

In [ ]:
for point in experiment.background:
    point.intensity.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('sepd')

### Gaussian widths

The background stays free while the widths join. diffraction-lib fixes it for this step and
leaves it fixed; here the fit reaches the same result with it free, as every other case does.
The Gaussian terms come first.

In [ ]:
experiment.peak.broad_gauss_sigma_0.free = True
experiment.peak.broad_gauss_sigma_1.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('sepd')

### Lorentzian width

Then the Lorentzian term.

In [ ]:
experiment.peak.broad_lorentz_gamma_1.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('sepd')

### Displacement

In [ ]:
structure.atom_sites['Si'].adp_iso.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('sepd')

### Peak decay

The back-to-back exponential decay terms β₀ and β₁.

In [ ]:
experiment.peak.decay_beta_0.free = True
experiment.peak.decay_beta_1.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('sepd')

In [ ]:
project.display.fit.correlations()

In [ ]:
project.display.structure('si')

## Save the project

In [ ]:
project.save_as('pd-neut-tof_si-sepd')
project.report.save_html('pd-neut-tof_si-sepd.html')

## Your own measured data

Measured data can also come from a file on your own disk. `add_from_data_path` reads a file of
time-of-flight, intensity and uncertainty columns into a new experiment, here in a project of its
own; the experiment then needs its instrument, profile and linked structure, as above. The file
here is SEPD's pattern from the collection, saved to disk first.

In [ ]:
data_path = edi.download_data('pd-neut-tof_si-sepd-sepd-data', destination='data')
own = edi.Project(name='own_sepd')
own.experiments.add_from_data_path(
    name='mine',
    data_path=data_path,
    sample_form='powder',
    beam_mode='time-of-flight',
    radiation_probe='neutron',
)
own.display.pattern('mine')